# 📖 Notebook 1: Why Service Discovery?

In a microservices world, instances come and go — they scale, crash, restart on new IPs.
Hard-coding addresses in config does not survive this.

**Service discovery** = a runtime phone book.
Services **register** when they start; clients **look them up** by name.

Two styles:
1. **Client-side** — the client asks the registry, then calls the instance directly.
2. **Server-side** — the client calls a load balancer / router which asks the registry.

## 🛠️ Setup

```bash
cd 05-microservices/service-discovery
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## A tiny in-memory registry

In [ ]:
import time, random

class Registry:
    """Maps service_name -> list of (host, port, last_heartbeat)."""
    def __init__(self, ttl=5):
        self.services = {}
        self.ttl = ttl

    def register(self, name, host, port):
        self.services.setdefault(name, {})[(host, port)] = time.time()
        print(f'registered {name} @ {host}:{port}')

    def heartbeat(self, name, host, port):
        self.services[name][(host, port)] = time.time()

    def deregister(self, name, host, port):
        self.services.get(name, {}).pop((host, port), None)

    def healthy_instances(self, name):
        now = time.time()
        return [addr for addr, hb in self.services.get(name, {}).items()
                if now - hb < self.ttl]

r = Registry()
r.register('orders', '10.0.0.1', 8080)
r.register('orders', '10.0.0.2', 8080)
print('healthy:', r.healthy_instances('orders'))


## Client-side discovery: pick an instance yourself

In [ ]:
def client_side_call(registry, service, path):
    instances = registry.healthy_instances(service)
    if not instances:
        raise RuntimeError('no instances for ' + service)
    host, port = random.choice(instances)  # naive load-balancing
    print(f'calling http://{host}:{port}{path}')
    return {'status': 200}

client_side_call(r, 'orders', '/v1/orders/42')


👉 Notebook 2 adds server-side discovery and shows what happens when an instance crashes.